# RecSys МАГОЛЕГО, ФКН ВШЭ

## Домашняя работа 1: Матричные алгоритмы рекомендаций

### Оценивание и штрафы
Всего заданий: **7**, максимальная оценка — **9 баллов**.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом пользоваться запрещается,даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке (например, лишние циклы, `np.vectorize`, `np.apply_along_axis` и так далее). Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

Использование генеративных языковых моделей разрешено только в случае явного указания на это. Необходимо прописать (в соответствующих пунктах, где использовались, либо в начале/конце работы):

- какая языковая модель использовалась
- какие использовались промпты и в каких частях работы
- с какими сложностями вы столкнулись при использовании генеративных моделей, с чем они помогли больше всего
  
Copy-paste ответа генеративной модели запрещается (кроме графиков, но все равно нужно явно прописывать использование)

**Дедлайн: 25.02.2026 23:59 (по МСК)**

**Сдавать сюда: [Классрум](https://classroom.google.com/c/ODQzNjI1ODIzMDEy/a/ODQzNjI1NzMzMjgz/details)**

### О задании
В данном домашнем задании вы реализуете алгоритмы рекомендаций на основе матричных разложений и сравните их между собой на датасете Yahoo! Movies.

P.S Пожалуйста, аккуратно оформляйте графики, ориентироваться можно на [это](https://github.com/esokolov/ml-course-hse/blob/master/2022-fall/seminars/sem02-charts.ipynb). У графиков обязательно должно быть:

- Название
- Подписанные оси
- Легенда, если необходимо (например, если несколько графиков на одном рисунке)
- Все должно быть четко видно и ничего не сливаться
- За некрасивые графики можем снизить баллы

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Структура датасета

Датасет можете скачать по [ссылке](https://drive.google.com/file/d/1PsAL83MQnvQuTpjrNXs8CiPpOeSGcUL-/view?usp=share_link)

Датасет состоит из 6 основных файлов. Все данные представлены в текстовом формате, где колонки разделены символом табуляции `\t`.

### 1. Оценки пользователей
В этих файлах содержатся оценки фильмов пользователей
-  **Файлы:** `ydata-ymovies-user-movie-ratings-train-v1_0` и `ydata-ymovies-user-movie-ratings-test-v1_0`
- **Поля:**
  * User ID
  * Movie ID
  * Rating_13: от 1 (F) до 13 (A+)
  * Rating_5: упрощенная шкала от 1 до 5



### 2. Демография пользователей
Информация про аудиторию

- **Файл:** `ydata-ymovies-user-demographics-v1_0`
- **Поля:** 
  * User ID
  * Year of birth
  * Sex (`m`/`f`)

### 3. Описание фильмов
Метаданные фильмов
* **Файл:** `ydata-ymovies-movie-content-descr-v1_0`
* **Что внутри:** Название, синопсис, жанры, режиссеры, актеры, количество наград, средняя оценка критиков и даже ссылки на постеры.
* **Важно:** Если данных нет, стоит заглушка `\N`. Если в поле несколько значений (например, список актеров), они разделены символом `|`.

### 4. Списки соответствия 
Файлы `mapping-to-movielens` и `mapping-to-eachmovie` позволяют связать ID фильмов Yahoo с другими популярными датасетами (MovieLens и EachMovie). Это полезно, если вы хотите объединить данные из разных источников


**Задание 0 (0.25 баллов):** Загрузите необходимые данные и постройте `use-item` матрицу интеракций $X$

In [ ]:
# Your code her 👋≧◉ᴥ◉≦

## Метрики

Пусть
- $K$ - длина списка рекомендаций (например, 10)
- $Rec_K$ - список из $K$ рекомендованных объектов
- $Rel$ — множество всех релевантных объектов для пользователя
- $rel_i$ - релевантность объекта на позиции $i$ (обычно 1, если объект релевантен, и 0, если нет)

В качестве метрик рекомендаций возьмем:

- $Precision@K = \frac{|Rec_K \cap Rel|}{K}$
  
- $Recall@K = \frac{|Rec_K \cap Rel|}{|Rel|}$ 

- $AP@K = \frac{1}{\min(K, |Rel|)} \sum\limits_{i = 1}^K rel_i \cdot \left(\frac{|Rec_i \cap Rel|}{i}\right)$
  
- $NDCG@K$:
  
  - $DCG@K = \sum\limits_{i = 1}^K \frac{rel_i}{\log_2(i + 1)}$
  
  - $IDCG@K = \sum\limits_{i = 1}^{\min(K, |Rel|)} \frac{1}{\log_2(i + 1)}$
  
  - $NDCG@K = \frac{DCG@K}{IDCG@K}$

**Задание 1 (0.5 баллов):** Реализуйте вышеперечисленные метрики для одного пользователя


In [ ]:
def precision_at_k(
    recommended_list: np.ndarray,
    relevant_items: np.ndarray,
    k: int = 10,
) -> float:
    """
    Calculate Precision@K.

    Parameters
    ----------
    recommended_list : array_like
        Ordered list of recommended item IDs.
    relevant_items : array_like
        List or set of relevant item IDs.
    k : int, optional
        The maximum number of recommendations to consider. Default is 10.

    Returns
    -------
    float
        The Precision@K score.
    """
    # Your code her 👋≧◉ᴥ◉≦
    raise NotImplementedError("Implement precision_at_k function")


def recall_at_k(
    recommended_list: np.ndarray,
    relevant_items: np.ndarray,
    k: int = 10,
) -> float:
    """
    Calculate Recall@K.

    Parameters
    ----------
    recommended_list : array_like
        Ordered list of recommended item IDs.
    relevant_items : array_like
        List or set of relevant item IDs.
    k : int, optional
        The maximum number of recommendations to consider. Default is 10.

    Returns
    -------
    float
        The Recall@K score.
    """
    # Your code her 👋≧◉ᴥ◉≦
    raise NotImplementedError("Implement recall_at_k function")


def ap_at_k(
    recommended_list: np.ndarray,
    relevant_items: np.ndarray,
    k: int = 10,
) -> float:
    """
    Calculate Average Precision (AP) at K.

    Parameters
    ----------
    recommended_list : array_like
        Ordered list of recommended item IDs.
    relevant_items : array_like
        List or set of relevant item IDs.
    k : int, optional
        The maximum number of recommendations to consider. Default is 10.

    Returns
    -------
    float
        The Average Precision score.
    """
    # Your code her 👋≧◉ᴥ◉≦
    raise NotImplementedError("Implement ap_at_k function")


def ndcg_at_k(
    recommended_list: np.ndarray,
    relevant_items: np.ndarray,
    k: int = 10,
) -> float:
    """
    Calculate Normalized Discounted Cumulative Gain (NDCG) at K.

    Parameters
    ----------
    recommended_list : array_like
        Ordered list of recommended item IDs.
    relevant_items : array_like
        List or set of relevant item IDs.
    k : int, optional
        The maximum number of recommendations to consider. Default is 10.

    Returns
    -------
    float
        The NDCG@K score.
    """
    # Your code her 👋≧◉ᴥ◉≦
    raise NotImplementedError("Implement ndcg_at_k function")


In [ ]:
def evaluate_recommender(
    recommended_list: np.ndarray,
    relevant_items: np.ndarray,
    k: int = 10,
) -> dict:
    """
    Evaluate recommender system using multiple metrics.

    Parameters
    ----------
    recommended_list : array_like
        Ordered list of recommended item IDs.
    relevant_items : array_like
        List or set of relevant item IDs.
    k : int, optional
        The maximum number of recommendations to consider. Default is 10.

    Returns
    -------
    dict
        A dictionary containing Precision@K, Recall@K, AP@K, and NDCG@K scores.
    """
    # Your code her 👋≧◉ᴥ◉≦
    raise NotImplementedError("Implement evaluate_recommender function")

## Baseline

Начнем с двух базовых алгоритмов рекомендаций:

- Случайный: рекомендуем случайные $K$ фильмов
- Топ популярных: рекомендуем $K$ самых популярных фильмов

**Задание 2 (0.25 баллов):** Имплементируйте алгоритмы рекомендаций выше и провалидируйте на тестовом датасете помощью с `evaluate_recommender`. Сделайте график метрик в зависимости от $K$ (и для всех методов ниже так же нужно сделать)

**Ответ:**

In [ ]:
class RandomRecommender:
    def __init__(self, k: int = 10) -> None:
        self.k = k

    def predict(self, user_seen_items: np.ndarray, k=10) -> np.ndarray:
        """Recoommend k random items not seen by the user."""
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement predict method")


In [ ]:
class MostPopularRecommender:
    def __init__(self, train_df: pd.DataFrame) -> None:
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement predict method")

    def predict(self, user_seen_items: np.ndarray, k: int = 10) -> np.ndarray:
        """Recommend k most popular items not seen by the user."""
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement predict method")

In [ ]:
# Your code her 👋≧◉ᴥ◉≦

## SVD (Singular Value Decomposition)

Давайте вспомним метод SVD

SVD в контексте рекомендательных систем часто формулируется как задача оптимизации матричной факторизации, чтобы справиться с разреженностью матрицы взаимодействий $R$

Вместо классического SVD, требующего полной матрицы, мы ищем низкоранговое приближение $R \approx P Q^T$, где $P$ и $Q$ — матрицы скрытых факторов пользователей и объектов

**Задача оптимизации:**
Мы минимизируем ошибку только по наблюдаемым оценкам $(u, i) \in \Omega$ с добавлением L2-регуляризации для предотвращения переобучения:
$$\min_{P,Q} \sum_{(u,i) \in \Omega} (R_{ui} - p_u^T q_i)^2 + \lambda (\|p_u\|^2 + \|q_i\|^2)$$

**Алгоритм обучения:**
Для решения этой задачи обычно используется стохастический градиентный спуск (SGD). Правила обновления весов для каждого наблюдения $R_{ui}$ выглядят следующим образом:
$$p_u \leftarrow p_u + \eta (e_{ui} q_i - \lambda p_u)$$
$$q_i \leftarrow q_i + \eta (e_{ui} p_u - \lambda q_i)$$

где $e_{ui} = R_{ui} - p_u^T q_i$ — ошибка предсказания

**Задание 3 (2 балла):** Реализуйте алгоритм рекомендаций на основе SVD. Лучше ли по сравнению с бейзлайном? Проанализируйте ближайших соседей в пространстве эмбеддингов. Переберите несколько размерностей латентного пространства

**Ответ:**

In [ ]:
class SVDRecommender:
    """
    SVD-based recommender using Truncated SVD for matrix factorization.
    """

    def __init__(self, hidden_dim: int = 50) -> None:
        """
        Initialize the model.

        Parameters
        ----------
        hidden_dim : int
            Number of latent factors (components) to keep.
        """
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement __init__ method")

    def fit(self, X: np.ndarray) -> None:
        """
        Fit the model.

        Parameters
        ----------
        X : np.ndarray or sparse matrix
            User-item interaction matrix.
        """
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement fit method")

    def predict(self, X_user: np.ndarray) -> np.ndarray:
        """Predict scores."""
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement predict method")

In [ ]:
# Your code her 👋≧◉ᴥ◉≦

## SLIM (Sparse Linear Methods)

Хотя SVD и матричная факторизация являются фундаментом рекомендательных систем, у них есть свои ограничения. Латентные факторы (скрытые признаки) часто сложны для интерпретации, а качество рекомендаций может страдать на очень разреженных данных, где классические меры сходства нестабильны

SLIM предлагает альтернативный взгляд: вместо проекции в скрытое пространство, давайте выучим матрицу связей **Item-Item** напрямую из данных. Это позволяет явно моделировать распространение интереса от просмотренных товаров к рекомендованным

SLIM — это линейная модель, которая обучается находить матрицу **разреженных** коэффициентов $B$ для прямого предсказания оценок на основе истории взаимодействий. Идея заключается в том, что рекомендации генерируются путем распространения интереса между похожими объектами

**Принцип работы:**
Предсказанный скор для пользователя рассчитывается как $\hat{X} = X B$, где $X$ — вектор истории пользователя

**Задача оптимизации:**
Обучение сводится к задаче ElasticNet-регрессии. Мы минимизируем следующую функцию потерь:
$$\mathcal{L}(B) = \frac{1}{2} \| X - X B \|_F^2 + \lambda_1 \| B \|_1 + \frac{\lambda_2}{2} \| B \|_F^2$$

**Ограничения (Constraints):**
Модель накладывает два важных ограничения:
1.  $B_{ii} = 0$: Диагональные элементы должны быть равны нулю (чтобы исключить тривиальное решение $B=I$, когда предмет рекомендует сам себя)
2.  $B_{ij} \ge 0$: Веса должны быть неотрицательными, что обеспечивает интерпретируемость (положительная связь между предметами)

**Преимущества:**
* Изучает отношения "Item-Item" напрямую из данных
* Создает разреженную и интерпретируемую матрицу

**Задание 4 (2 балла):** Теперь реализуем модель SLIM. Все так же нужно сравнить с предыдущими моделями. Проанализируйте item-item коэффициенты. Получается ли матрица разреженной?

**Ответ:**

In [ ]:
class SLIMRecommender:
    """
    Sparse Linear Methods (SLIM) recommender.
    Learns a sparse item-item similarity matrix W via ElasticNet regression.
    """

    def __init__(self, l1_ratio: float = 0.1, alpha: float = 1.0) -> None:
        """
        Initialize SLIM parameters.

        Parameters
        ----------
        l1_ratio : float
            Mixing parameter for ElasticNet (0 for Ridge, 1 for Lasso).
        alpha : float
            Regularization strength.
        positive : bool
            Whether to enforce non-negativity constraint on weights.
        """
        self.l1_ratio = l1_ratio
        self.alpha = alpha
        self.W = None

    def fit(self, X: np.ndarray) -> None:
        """
        Fit the model.

        Parameters
        ----------
        X : np.ndarray or sparse matrix
            User-item interaction matrix.
        """
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement fit method")

    def predict(self, X_user: np.ndarray) -> np.ndarray:
        """Predict scores."""
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement predict method")

## EASE (Embarrassingly Shallow AutoEncoders)

SLIM создает качественную и разреженную матрицу, но платит за это высокую цену в производительности. Обучение модели требует итеративного решения множества задач регрессии (Lasso/ElasticNet), что вычислительно дорого и медленно. Это ограничивает масштабируемость метода на больших каталогах

EASE решает эту проблему радикально. Убрав L1-регуляризацию и ограничение на неотрицательность весов, мы можем свести задачу к аналитической формуле. Это позволяет получить точное решение за один шаг, избегая долгого градиентного спуска.

EASE — это упрощенный линейный автоэнкодер, который, в отличие от SLIM, имеет замкнутое аналитическое решение и не требует итеративного обучения (градиентного спуска)

**Задача оптимизации:**
Функция потерь похожа на SLIM, но используется только L2-регуляризация, и убрано ограничение на неотрицательность весов (остается только ограничение на нулевую диагональ):
$$\min_{B} \frac{1}{2} \| X - X B \|_F^2 + \frac{\lambda}{2} \| B \|_F^2, \quad \text{при } \text{diag}(B) = 0$$

**Аналитическое решение:**
Решение вычисляется через обращение матрицы Грама. Пусть $H = (X^T X + \lambda I)^{-1}$. Тогда матрица весов $B$ вычисляется по формуле:

$$B_{ij} = \begin{cases} 0, & \text{если } i = j \\ -\frac{H_{ij}}{H_{jj}}, & \text{если } i \neq j \end{cases}$$

**Особенности:**
Метод очень быстр в обучении для матриц среднего размера и часто показывает качество лучше, чем сложные нейросети, но матрица $B$ получается плотной (dense), что затрудняет масштабирование

**Задание 5 (2 балла):** Реализуем EASE алгоритм. Продолжаем сравнивать с предыдущими методами. Насколько быстрее SLIM?

**Ответ:**

In [ ]:
class EASERecommender:
    """
    Embarrassingly Shallow AutoEncoders (EASE).
    Computes a closed-form linear solution using the inverse of the Gram matrix.
    """

    def __init__(self, l2_lambda: float) -> None:
        """
        Initialize EASE parameters.

        Parameters
        ----------
        l2_lambda : float
            L2 regularization strength.
        """
        self.l2_lambda = l2_lambda
        self.W = None

    def fit(self, X: np.ndarray) -> None:
        """
        Fit the model.

        Parameters
        ----------
        X : np.ndarray or sparse matrix
            User-item interaction matrix.
        """
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement fit method")

    def predict(self, X_user: np.ndarray) -> np.ndarray:
        """Predict scores."""
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement predict method")

In [ ]:
# Your code her 👋≧◉ᴥ◉≦

## SANSA (Scalable Approximate NonSymmetric Autoencoder)

EASE обучается очень быстро и дает высокое качество, но сталкивается с "бутылочным горлышком" при росте количества товаров. Матрица весов $B$ в EASE получается плотной  — в ней почти нет нулей. Если у нас десятки тысяч товаров, хранение и умножение такой матрицы требует огромного количества памяти ($O(n^2)$) и времени.

Здесь на сцену выходит SANSA. Этот метод сохраняет математическую логику EASE, но использует приближенные вычисления (разреженное разложение Холецкого), чтобы итоговая матрица весов оставалась разреженной, как в SLIM. Это позволяет масштабировать подход EASE на огромные каталоги.

SANSA решает главную проблему EASE — масштабируемость.
В EASE матрица $B$ является плотной, что приводит к огромным затратам памяти и времени при большом количестве предметов. SANSA предлагает метод для эффективного приближения EASE с сохранением разреженности матрицы весов.

**Метод приближения:**
Вместо явного обращения огромной матрицы $P = X^T X + \lambda I$, SANSA использует разреженное разложение Холецкого:
$$P \approx L D L^T$$
где $L$ — нижняя треугольная разреженная матрица.

**Вычисление весов:**
Приближенная обратная матрица выражается через решение систем линейных уравнений с разреженными треугольными матрицами, что позволяет избежать плотных вычислений. Итоговая матрица весов $B$ получается разреженной, что позволяет работать с каталогами из десятков и сотен тысяч товаров.

**Задание 6 (2 балла):** Воспользуйтесь библиотекой [SANSA](https://pypi.org/project/sansa/) и финализируйте эксперименты метриками этой модели

**Ответ:**

In [ ]:
class SANSARecommender:
    """
    Scalable Approximate NonSymmetric Autoencoder (SANSA).
    Approximates the EASE solution while enforcing sparsity in the weight matrix.
    """

    def __init__(self, l2_lambda=100, density_target=0.05) -> None:
        """
        Initialize SANSA parameters.

        Parameters
        ----------
        l2_lambda : float
            L2 regularization strength.
        density_target : float
            Target density for the weight matrix (e.g., 0.05 means 5% non-zero).
        """
        self.l2_lambda = l2_lambda
        self.density_target = density_target
        self.W = None

    def fit(self, X: np.ndarray) -> None:
        """
        Fit the model.

        Parameters
        ----------
        X : np.ndarray or sparse matrix
            User-item interaction matrix.
        """
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement fit method")

    def predict(self, X_user: np.ndarray) -> np.ndarray:
        """Predict scores."""
        # Your code her 👋≧◉ᴥ◉≦
        raise NotImplementedError("Implement predict method")

In [ ]:
# Your code her 👋≧◉ᴥ◉≦

## Conclusion

Опишите выводы ваших экспериментов:
